# Attention Mechanism


### 1. Simple self-attention mechanism without trainable weights

Context vectors play a crucial role in self-attention. Their purpose is to create enriched representations of each element in an input sequence (like a sentence) by incorporating information from all other elements in the sequence. This is essential in LLMs, which need to understand the relationship and relevance of words in a sentence to each other. Later we will learn to construct these context vectors so that they are relevant to generate the next token.

In [3]:
import torch

from src.Attention.SelfAttentionV2 import SelfAttentionV2

# example input
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]]  # step
)

The first step is to calculate the intermediate attention scores

In [5]:
query = inputs[1] # query is the embedding of the word "Journey" since it is at index 1
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i)

print("Computed attention scores:")
print(attn_scores_2)

Computed attention scores:
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Note: The dot product is a measure of similarity since it quantifies how closely two vectors are aligned.

In [7]:
# Normalize the scores
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum: ", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum:  tensor(1.0000)


This works; however, in practice, it is more common and advisable to use the softmax function for normalization. It offers a better approach for managing extreme values and better gradient properties during training. Here is a softmax normalization:

In [8]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum: ", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


The softmax function ensures that the attention weights are always positive. This makes the output interpretable as probabilities or relative importance.

What do these numbers represent? The attention weights determine the relative importance of each token ("Your", "journey", "starts", "with", "one", "step") relative to the query token "journey." You will notice there are 6 attention weights since there are six input tokens in the original input sequence.

We should prefer to use the softmax function provided by torch to ensure numerical stablity and proper handling of under/overflowing small/large values.

In [9]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum: ", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)


Now we are ready to compute the context vector by multiplying embedding input tokens with the corresponding attention weights and then summing the resulting vectors. Thus, the context vector is the weighted sum of all the input vectors multiplied by their corresponding attention weights.

In [12]:
query = inputs[1] # query is the embedding of the word "Journey" since it is at index 1
context_vector_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs): # x_i is the embedding of the input token at index i
    context_vector_2 += attn_weights_2[i] * x_i

print("Context vector:")
print(context_vector_2)

Context vector:
tensor([0.4419, 0.6515, 0.5683])


Next, we will generalize this procedure for computing context vectors to calculate all context vectors simultaneously. Said another way, we want to compute the attention weights for every input token, not just "journey" like we used above. This will enable us to compute all context vectors at once.

In [13]:
attn_scores = torch.empty(6, 6) # six input tokens will have six attention weights. The tensor will have 6 * 6 = 36 elements. 6 input tokens, 6 attention weights per input (query) token.
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print("Computed attention scores:")
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


For loops in python are extremely slow and inefficient. Instead, we can achieve the same result with matrix multiplication directly.

In [14]:
attn_scores = inputs @ inputs.T
print("Computed attention scores:")
print(attn_scores)

Computed attention scores:
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Again, we need to normalize our attention scores.

In [17]:
attn_weights = torch.softmax((attn_scores), dim=-1)
print("Attention weights:")
print(attn_weights)
print("All Row Sum: ", attn_weights.sum(dim=-1))

Attention weights:
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
All Row Sum:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [18]:
all_contex_vecs = attn_weights @ inputs
print("All context vectors:")
print(all_contex_vecs)

All context vectors:
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


Notice that the 2nd row values match our original single row computation from earlier.


### 2. Implementing self-attention with trainable weights


Now that we understand how self-attention works, we will now implement self-attention with trainable weights that builds on the previous concepts. We will see there is only a slight difference from our first implementation. The weight matricies are crucial so that the model (the attention module inside the model) can learn to produce "good" context vectors.

To begin our analysis, again we will focus on computing a single context vector using the input token at index 1 (aka the input token "journey").

In [19]:
x_2 = inputs[1] # the embedding of the word "Journey" since it is at index 1
d_in = inputs.shape[1] # dimension of the input embedding, d_in = 3
d_out = 2 # output embedding, d_out = 2

Note: usually the input and output dimensions are usually the same, but to make the example eaiser to follow we use different dimensions.

In [22]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

We set requires gradient to false to reduce clutter in the outputs, but if we were to use the weight matrices for model training, we would set to true.

#### Why query, key, and value?

The terms are borrowed from the domain of information retrieval and databases, where similar concepts are used to store, search, and retrieve information.

A query is analogous to a search query in a database. It represents the current item the (e.g., a word or token in a sentence) the model focuses on or tries to understand. The query is used to prob the other parts of the input sequence to determine how much attention to pay to them.

The key is like a database key used for indexing and searching. In the attention mechanism, each item in the input sequence has an associated key. These keys are used to match the query.

The value in this context is similar to the value in a key-value pair in a database. It represents the actual content or representation of the input items. Once the model determines which keys (and thus which pars of the input) are most relevant to the query (the current focus item), it retrieves the corresponding values.

In [23]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


Even though our temporary goal is only to compute the one context vector, we still require the key and value vectors for all input elements are they are involved in computing the attention weights with respect to the query. We obtain all keys and values via matrix multiplication:

In [24]:
keys = inputs @ W_key
values= inputs @ W_value
print("Keys shape: ", keys.shape)
print("Values shape: ", values.shape)

Keys shape:  torch.Size([6, 2])
Values shape:  torch.Size([6, 2])


We can see we successfully project the six input tokens from a three-dimensional space to a two-dimensional space.

Now, let's compute the attention score:

In [26]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print("Attention score:")
print(attn_score_22)

Attention score:
tensor(1.8524)


The resulting score is unnormalized. Again, we can generalize this computation to all attention scores via matrix multiplication:

In [27]:
attn_scores_2 = query_2 @ keys.T
print("Attention scores:")
print(attn_scores_2)

Attention scores:
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


Now, we want to go from the attention scores to the attention weights. We compute the attention wieghts by scaling the attention scores and using the softmax function. However, now we scale the attention scores by dividing them by the square root for the embedding dimension of the keys (taking the square root is mathematically equivalent as exponentiation by 0.5):

In [28]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print("Attention weights:")
print(attn_weights_2)

Attention weights:
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])



#### The rationale behind scaled-dot product attention:

The reason for normalizing by the embedding dimensions size is to improve training performance by avoiding small gradients. For instance, large embedding dimensions, large dot products can result in very small gradients during backprop due to the softmax function applied to them. As dot products increase, the softmax function behaves more like a step function (implications of precision), resulting in gradients nearing zero. These small gradients can drastically slow down learning or cause training to stagnate.  The scaling by the square root of the embedding dimensions is the reason why this self-attention mechanism is also called the scaled dot product attention.

Similar to when we compute the context vector as a weighted sum over the input vectors, we now compute the context vector as a weighted sum over the value vectors. Here the attention weights serve as a weighting factor that weights the respective importance of each value vector.

In [29]:
context_vector_2 = attn_weights_2 @ values
print("Context vector:")
print(context_vector_2)

Context vector:
tensor([0.3061, 0.8210])


So far, we have only computed a single context vector. Next, we will generalize the code to compute all context vectors in the input sequence.

In [30]:
from src.Attention.SelfAttentionV1 import SelfAttentionV1
torch.manual_seed(123)
sa_v1 = SelfAttentionV1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Notice that the second row matches the context vector from our previous section above. The advantage of this attention implementation is that the query, key, and value matrices are all trainable which enable us to improve the attention and retrieval results.

We can improve our implementation further by utilizing PyTorch's nn.Linear layers, which effectively preforms matrix multiplications when the bias untis are disabled. Additionally, a significant advantage of using linear instead of manually implementing nn.Parameter is that Linear has an optimized weigh initialization scheme, contributing to more stable and effective model training.

In [32]:
from src.Attention.SelfAttentionV2 import SelfAttentionV2
torch.manual_seed(789)
sa_v2 = SelfAttentionV2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Note: outputs are different becuase they use different initial weights for the weight matrices.

Next, we will focus on casual attention which prevents the model from accessing future information in the sequence, which is crucial to prevent lookahead bias in a task specifically designed to predict the next word.


### 3. Implementing casual self-attention

Casual attention, also known as "masked attention", is a specialized form of self-attention.... To achieve GPT-like LLMs, for each token processed, we mask the future tokens, which come after the current token in the input text. We mask out the attention weights above the diagonal, and we normalize the nonmasked attention weights such that the attention weights sum to 1 in each row.

In [33]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print("Attention weights:")
print(attn_weights)

Attention weights:
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Using the lower triangle matrix we can mask our attention weights.

In [36]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


We can use this matrix to zero-out the values above the diagnoal.

In [37]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


The third step is to normalize the attention weights to sum up to 1 again in each row. We can achieve this by dividing each element in each row by the sum in each row:

In [38]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


While we could wrap up our implementation of causal attention at this point, we can still improve it thanks to a mathematical property of the softmax function. The softmax function converts its inputs into a probability distribution. When negative infinity values are present in a row, the softmax function treats them as zero probability. (Mathematically, this is because e^-inf approaches 0.)

We can implement this more efficient masking "trick" by creating a mask with 1s above the diagnoal and then replacing these 1s with negative infinity (-inf) values:

In [39]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


Now all we need to do is apply the softmax function to these masked results and we are done:

In [40]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print("Attention weights:")
print(attn_weights)

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


Now we could use the modified attention weights to compute the context vectors via:

In [41]:
context_vector = attn_weights @ values
print("Context vector:")
print(context_vector)

Context vector:
tensor([[0.1855, 0.8812],
        [0.2795, 0.9361],
        [0.3133, 0.9508],
        [0.2994, 0.8595],
        [0.2702, 0.7554],
        [0.2772, 0.7618]], grad_fn=<MmBackward0>)


We will cover one minor tweak to the causal attention mechanism that is useful for reducing overfitting when training LLMs.

#### Masking additional attention weights with dropout

Dropout is only used during training and is disabled after. Dropout is typically applied at two specific times. After calculating the attention weights or after applying attention weights to the value vectors. Here we will apply dropout mask after computing the attention weights because it's the more common variant in practice.

We will use a dropout rate of 50% for illustration but typically dropout rate is set closer to 0.1 or 0.2.

In [43]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(p=0.5)
example = torch.ones(6, 6) # Matrix of ones
print("Dropout example:")
print(dropout(example))

Dropout example:
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


As we see, approximately half the values are zeroed out. To compensate for the reduction in active elements, the values of the remaining elements in the matrix are scaled up by a factor of 1/0.5 = 2. This scaling is crucial for maintaining the overall balance of the attention weights ensuring that the average influence of the attention mechanism remains consistent during both training and inference phases.

Now, let's apply dropout to the attention weight matrix itself:

In [44]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


Now we will develop a concise Python self-attention class designed to facilitate the efficient application of these two techniques: (1) causal attention and dropout modification.

Before we begin, let's ensure that the code can handle batches consisting of more than one input so that the CausalAttention class supports the batch output produced by the data loader we implemented in Chapter 2 (Data Processing).

In [45]:
batch = torch.stack((inputs, inputs), dim=0)
print("Batch shape:" ,batch.shape)
# two inputs with six tokens each; each token has embedding in 3 dimensions

Batch shape: torch.Size([2, 6, 3])


The result is a three dimensional tensor consisting of two input texts with six tokens each, where each token is a three-dimensional embedding vector.

We can use the CausalAttention class as follows:

In [47]:
from src.Attention.CausalAttention import CausalAttention
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0)
print("Context vector shape:")
print(ca(batch).shape)

Context vector shape:
torch.Size([2, 6, 2])


So far, we have focused on Causal attention in neural networks. Next we will expand on this concept and implement a multi-head attention module that implements several causal attention mechansims in parallel.


### 4. Implementing multi-head casual self-attention

Our final step will be to extend the previously implemented causal attention class over multiple heads. This is called "multi-head attention."

Mult-head attention allows multiple heads to operate independently. In this context, a single causal attention module can be considered single-head attention, where there is only one set of attention weights processing the inputs sequentially.

We will tackle this expansion by first stacking multiple CausalAttention modules. Then we will implement the same multi-head attention module in a more complicated but computationally efficient way.

Using multiple instances of the self-attention mechanism can be computationally intensive, but it's critical for the kind of complex pattern recognition that models like transformer-based LLms are known for.... The main idea behind multi-head attention is to run the attention mechanism multiple times (in parallel) with different, learned linear projections--the result of multiplying the input data (like the query, key, and value vectors in attention mechanisms) by a weight matrix.

In [51]:
from src.Attention.MultiHeadAttentionWrapper import MultiHeadAttentionWrapper
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0, num_heads=2)

context_vecs = mha(batch)

print("Context vectors:")
print(context_vecs)
print("Context vector shape:", context_vecs.shape)

Context vectors:
tensor([[[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.5337, -0.1051,  0.5085,  0.3508],
         [-0.5323, -0.1080,  0.5084,  0.3508],
         [-0.5323, -0.1079,  0.5084,  0.3506],
         [-0.5297, -0.1076,  0.5074,  0.3471],
         [-0.5311, -0.1066,  0.5076,  0.3446],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
Context vector shape: torch.Size([2, 6, 4])


The first dimension is 2 since we have two input texts (the input texts are duplicated, which is why thet context vectors are exactly the same for those). The second dimension refers to the 6 tokens in each total text input. The third dimension refers to the four-dimensional embedding of each token (two attention heads and embedding dimension of 2 concatenated together into one context vector = 4.

This class works, but each head is processed sequentially in the forward method. We can improve our performance by enabling parallel data processing in the heads. One way to achieve this is by computing the outputs for all attention heads simultaneously via matrix multiplication.

#### Efficient Multi-Head Attention

Even though the MultiHeadAttention class may look mathematically complicated, the class implements the same concept as the Wrapper class, but instead uses matrix multiplication for efficiency.

In [52]:
from src.Attention.MultiHeadAttention import MultiHeadAttention
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0, num_heads=2)
context_vecs = mha(batch)

print("Context vectors:")
print(context_vecs)
print("Context vector shape:", context_vecs.shape)

Context vectors:
tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
Context vector shape: torch.Size([2, 6, 2])


We have now implemented an efficient multi-head attention class that we will use when we train the LLM. Note: while the code is fully functional, I used small embedding sizes and number of attention heads to keep the outputs readable.

For comparison, the smallest GPT-2 model (117 million parameters) has 12 attention heads and a context vector embedding size of 768. The largest GPT-2 model (1.5 billion parameters) has 25 attention heads and context vector embedding size of 1,600. The embedding sizes of the token inputs and the context embeddings are the ame in GPT models (d_in = d_out.)

#### In Summary:

- Attention mechanisms transform input elements into enhanced context vector representations that incorporate information about all inputs.
- In self-attention, the query, key, and value vectors are learned parameters that are used to compute the attention weights.
- Causal attention masks the future tokens in the input sequence, preventing the model from accessing future information.
- Multi-head attention allows multiple attention heads to operate independently.
- The MultiHeadAttention class implements efficient parallel processing of attention heads.
- In addition to causal attention masks to zero-out attention weights, we can add a dropout mask to reduce overfitting in LLMs.